# 第一章（一）：用音乐材料学习 Python 基础

这组示例与正文“数据、运算与容器”“流程控制”“函数、模块、路径与异常”三节配套，围绕音高、时值和音符事件展开，不读取外部音乐文件。内容包括：

- 核对 Notebook 实际使用的 Python 解释器、Jupyter 内核与当前工作目录；
- 读写基础类型、容器、条件、循环和函数；
- 使用 `pathlib.Path` 定位文件，并认读常见的 `os.path` 写法；
- 从异常回溯的最后一行识别错误类型，确定下一项核查。

首次运行时，先选择由第一章虚拟环境启动的内核，再依次执行 **Restart Kernel** 和 **Run All**。

## 1. 解释器、Jupyter 内核与当前工作目录

Python 解释器负责执行代码；虚拟环境为项目提供独立的解释器入口和包安装位置；Jupyter 内核则运行代码并与 Notebook 前端通信。在 Python 内核中，`sys.executable` 给出当前进程使用的解释器路径，可据此判断内核是否连接到预期环境。`Path.cwd()` 返回当前工作目录（current working directory, cwd），相对路径从这里开始解释。

In [ ]:
import importlib
import os
import sys
import warnings
from pathlib import Path

print(f"Python 版本：{sys.version.split()[0]}")
print(f"解释器路径：{sys.executable}")
print(f"当前工作目录：{Path.cwd().resolve()}")

Notebook 启动时的 cwd 可能是项目根目录，也可能是其中的某个子目录。下面从 cwd 逐级向上寻找同时包含 `CODE/` 和 `BOOK/` 的目录：找到后用 `break` 结束循环；到达文件系统根目录仍未找到时抛出错误。在常见的 Notebook 执行环境中，代码单元没有可用于定位 `.ipynb` 文件的 `__file__`，因此不能照搬普通脚本的定位方式。

In [ ]:
search_dir = Path.cwd().resolve()

while True:
    if (search_dir / "CODE" / "chapter01").is_dir() and (search_dir / "BOOK").is_dir():
        PROJECT_ROOT = search_dir
        break
    if search_dir.parent == search_dir:
        raise FileNotFoundError("未找到同时包含 CODE/chapter01 和 BOOK 的项目根目录")
    search_dir = search_dir.parent

CHAPTER_DIR = PROJECT_ROOT / "CODE" / "chapter01"
print(f"项目根目录：{PROJECT_ROOT}")
print(f"本章代码目录：{CHAPTER_DIR}")

if str(CHAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIR))

from note_events import make_note_events

### 顶部配置区

影响全局结果的路径、采样率、开关和随机种子等配置通常集中放在 Notebook 开头。全大写名称沿用 Python 的常量命名约定，表示代码通常不会重新绑定该名称；Python 并不禁止重新赋值，也不会使列表或字典自动变成不可变对象。把 `TRANSPOSE_SEMITONES` 改为其他整数后，应重新从头运行，避免新旧状态混用。

In [ ]:
TRANSPOSE_SEMITONES = 0
LONG_NOTE_THRESHOLD_BEATS = 1.0

print(f"移调：{TRANSPOSE_SEMITONES:+d} 个半音")

## 2. 值、变量、类型与运算

Python 使用对象表示和处理数据。对象具有确定的类型和值，赋值语句把变量名绑定到对象；同一个名称以后可以重新绑定到另一个对象。`int`、`float`、`str`、`bool` 和 `None` 是后续代码中常见的基础对象，`type()` 可用于核对对象的实际类型。`None` 是 `NoneType` 的唯一对象，通常表示缺少值或尚无结果；它与数值 0、空字符串和 `False` 的含义均不相同。判断一个对象是否为 `None` 时，惯用 `is None` 或 `is not None`。

In [ ]:
base_pitch = 60
pitch = base_pitch + TRANSPOSE_SEMITONES
duration_beats = 1.5
note_name = "C4"
is_accented = True
lyric = None

values = [pitch, duration_beats, note_name, is_accented, lyric]
for value in values:
    print(f"{value!r:>6} -> {type(value).__name__}")

算术运算符可用于音程、拍数和比例计算。对于这里使用的整数和浮点数，`/` 执行真除法，两个整数相除也会得到浮点数；`//` 返回向下取整后的商，并非一律截去小数部分；`%` 返回相应的余数；`**` 表示乘方。结果的音乐意义取决于输入单位和具体问题。

In [ ]:
lower_pitch = 60 + TRANSPOSE_SEMITONES
higher_pitch = 67 + TRANSPOSE_SEMITONES
interval_semitones = higher_pitch - lower_pitch
two_measures = 4 * 2
average_notes_per_beat = 9 / 4
complete_measures = 9 // 4
remaining_beats = 9 % 4
frequency_ratio = 2 ** (interval_semitones / 12)

print(f"音程：{higher_pitch} - {lower_pitch} = {interval_semitones} 个半音")
print(f"两个四拍小节：{two_measures} 拍")
print(f"9 个音分布在 4 拍：平均 {average_notes_per_beat:.2f} 个/拍")
print(f"9 拍包含 {complete_measures} 个完整四拍小节，余 {remaining_beats} 拍")
print(f"十二平均律下该音程的频率比约为 {frequency_ratio:.3f}")

比较表达式产生布尔值。组合多个条件时，`and` 表示条件同时成立，`or` 表示至少一个条件成立，`not` 对真假取反。`and` 与 `or` 从左向右短路求值：左侧已经足以确定结果时，不再计算右侧。成员检查写作 `in` 或 `not in`。

In [ ]:
c_major_pentatonic_classes = {0, 2, 4, 7, 9}
is_midi_pitch = 0 <= pitch <= 127
is_c_major_pentatonic = pitch % 12 in c_major_pentatonic_classes
needs_attention = (not is_midi_pitch) or duration_beats <= 0
is_usable_note = is_midi_pitch and duration_beats > 0

print(f"有效 MIDI 音高：{is_midi_pitch}")
print(f"属于 C 大调五声音阶音级集合：{is_c_major_pentatonic}")
print(f"需要检查：{needs_attention}")
print(f"可作为正时值音符处理：{is_usable_note}")

`pitch % 12` 把 MIDI 音符编号归入十二个音级，其中 C 对应 0，升 C 或降 D 对应 1，依次类推；上面的集合表示 C 大调五声音阶所用的音级，不区分八度。

`int()`、`float()` 和 `str()` 可以把合适的输入转换为相应类型。转换并非总能成功，例如 `int("C4")` 会产生 `ValueError`。字符串可用转义序列表示换行、制表等字符；f-string 将表达式的值嵌入字符串，`:.2f` 表示按小数点后两位显示，`:>3` 表示在至少 3 个字符宽的字段中右对齐。

In [ ]:
pitch_text = "64"
duration_text = "0.75"
converted_pitch = int(pitch_text)
converted_duration = float(duration_text)
summary = str(converted_pitch) + " / " + str(converted_duration)

print("音高\t时值")
print(f"{converted_pitch:>3}\t{converted_duration:.2f} 拍")
print(f"转换后拼接：{summary}")

## 3. 列表、元组、字典与集合

列表有顺序且可增删和修改；元组有顺序，元组本身不可变，常表示固定记录或函数的多个返回值；字典保存键值映射并按插入顺序遍历；集合用于去重和成员检查，不保证固定顺序。序列索引从 0 开始，负索引从末尾计数，切片的终止位置不包含在结果中。

In [ ]:
pentatonic_names = ["C", "D", "E", "G", "A"]
fixed_event = (0.0, 60 + TRANSPOSE_SEMITONES, 1.0)
pitch_name_by_number = {60: "C4", 62: "D4", 64: "E4"}
pitch_classes = {60 % 12, 62 % 12, 64 % 12, 72 % 12}

print(f"列表首项：{pentatonic_names[0]}")
print(f"列表末项：{pentatonic_names[-1]}")
print(f"前三项切片：{pentatonic_names[:3]}")
print(f"列表长度：{len(pentatonic_names)}")
print(f"固定事件（起始、音高、时值）：{fixed_event}")
print(f"字典索引：{pitch_name_by_number[60]}")
print(f"安全查询缺失键：{pitch_name_by_number.get(67, '未知音名')}")
print(f"去重后的音级集合：{pitch_classes}")
print(f"音级 7 不在集合中：{7 not in pitch_classes}")

赋值本身不会复制对象。两个名称若指向同一列表，通过其中一个名称原地修改列表，另一个名称看到的内容也会随之变化。`.copy()` 创建浅复制，使外层列表彼此独立；若列表中还包含列表、字典等可变对象，两个外层列表仍会共享这些内层对象。

In [ ]:
original_melody = [60, 62, 64]
same_list = original_melody
independent_list = original_melody.copy()
same_list[0] = 61

print(f"原列表：{original_melody}")
print(f"同一对象：{same_list}")
print(f"修改前复制的列表：{independent_list}")

下面的列表保存八个音符事件。`onset_beat` 与 `duration_beat` 的单位为拍；`pitch_midi` 是 0—127 的 MIDI 音符编号；`velocity_midi` 在本例中取 1—127。MIDI 1.0 的 Note On 力度数据范围为 0—127，其中 0 用于表示相应音符结束，因此实际起音使用 1—127。力度值常用于控制合成器的振幅或音色响应，却不是声学响度的直接测量值。四个字段均为必填项，不使用 `None` 表示缺失。

In [ ]:
NOTE_EVENTS = make_note_events(TRANSPOSE_SEMITONES)

first_event = NOTE_EVENTS[0]
print(f"第一个事件的音高：{first_event['pitch_midi']}")
print(f"不存在的歌词字段：{first_event.get('lyric')}")
for field, value in first_event.items():
    print(f"{field:>8}: {value}")

## 4. 条件与循环

Python 使用缩进划分代码块。同一层级的语句保持相同缩进；[PEP 8](https://peps.python.org/pep-0008/) 建议每级使用四个空格，并以空格作为首选缩进方式。缩进改变，语句所属的条件或循环也会随之改变。

In [ ]:
velocity_midi = NOTE_EVENTS[3]["velocity_midi"]

if velocity_midi >= 100:
    velocity_band = "高力度区间"
elif velocity_midi >= 70:
    velocity_band = "中等力度区间"
else:
    velocity_band = "低力度区间"

length_label = "长音" if NOTE_EVENTS[3]["duration_beat"] >= LONG_NOTE_THRESHOLD_BEATS else "短音"
print(f"力度 {velocity_midi}：{velocity_band}；时值分类：{length_label}")

例如，下面两段代码的最后一行位置不同：

```python
if velocity_midi >= 90:
    label = "较强"
    print(label)      # 只有条件成立才执行

if velocity_midi >= 90:
    label = "较强"
print(label)          # 已经离开 if 代码块
```

第一段的 `print` 属于 `if` 代码块，只在条件成立时执行；第二段的 `print` 已经离开条件语句。若条件不成立且此前没有给 `label` 赋值，第二种写法会引用尚未定义的名称。

In [ ]:
print("用 range 生成五个级数：")
for degree in range(1, 6):
    print(degree, end=" ")
print()

names = ["C4", "D4", "E4"]
durations = [1.0, 0.5, 0.5]
for number, (name, duration) in enumerate(
    zip(names, durations, strict=True), start=1
):
    print(f"{number:>2}. {name}: {duration:.2f} 拍")

print("只使用二元组中的音高，以下划线标记暂不使用的时值：")
for pitch_value, _ in [(60, 1.0), (62, 0.5), (64, 0.5)]:
    print(pitch_value, end=" ")
print()

`zip` 默认在最短的输入结束时停止，较长输入末尾的元素不会进入结果。这里的音名和时值应一一对应，因此使用 `strict=True`；长度不一致时，程序会抛出 `ValueError`，而不是悄然忽略多出的元素。

`while` 在条件为真时重复执行。`continue` 跳过本轮尚未执行的语句，重新判断循环条件；`break` 立即结束所在的最内层循环。循环应有清楚的结束路径：条件最终变为假，或者某条路径会执行 `break`。下面先推进索引再判断休止，因此 `continue` 不会使循环停留在同一位置。

In [ ]:
sequence_with_rests = [60, None, 62, 64, None, 67, 69]
index = 0
collected_pitches = []

while index < len(sequence_with_rests):
    current_pitch = sequence_with_rests[index]
    index += 1
    if current_pitch is None:
        continue
    collected_pitches.append(current_pitch)
    if len(collected_pitches) == 4:
        break

print(f"跳过休止并收集前四个音高：{collected_pitches}")

`sorted` 返回新的已排序列表，`key=` 指定比较键。这里的 `lambda event: ...` 创建一个简短的匿名函数：它接收一个音符事件，并返回由起始拍和音高组成的二元组。二元组先比较第一项，第一项相同时再比较第二项，因此事件先按起始拍、再按音高排列。

In [ ]:
events_by_time_and_pitch = sorted(
    NOTE_EVENTS,
    key=lambda event: (event["onset_beat"], event["pitch_midi"]),
)
event_pitches = [event["pitch_midi"] for event in events_by_time_and_pitch]
event_durations = [event["duration_beat"] for event in events_by_time_and_pitch]

print(f"按起始拍、音高排序后的前两个事件：{events_by_time_and_pitch[:2]}")
print(f"最低音：{min(event_pitches)}")
print(f"最高音：{max(event_pitches)}")
print(f"音符时值总和：{sum(event_durations):.2f} 拍")

`min` 和 `max` 在没有提供 `default` 时要求输入非空，`sum` 对空输入返回起始值，默认是 0。音符时值之和也不一定等于片段从开始到结束的持续时间：同时发声的音符会分别计入，休止则不会计入。

列表推导式的基本结构是 `[输出表达式 for 元素 in 可迭代对象 if 条件]`，末尾的筛选条件可以省略。字典推导式将输出写成 `键: 值`，集合推导式使用花括号并去除重复元素。圆括号中的生成器表达式按需产生元素；生成器对象不支持索引，迭代结束后也不会自动回到开头。

In [ ]:
long_pitches_loop = []
for event in NOTE_EVENTS:
    if event["duration_beat"] >= LONG_NOTE_THRESHOLD_BEATS:
        long_pitches_loop.append(event["pitch_midi"])

long_pitches = [
    event["pitch_midi"]
    for event in NOTE_EVENTS
    if event["duration_beat"] >= LONG_NOTE_THRESHOLD_BEATS
]
unique_pitches = {
    event["pitch_midi"] for event in NOTE_EVENTS
}
pitch_counts = {
    pitch_value: sum(
        1 for event in NOTE_EVENTS
        if event["pitch_midi"] == pitch_value
    )
    for pitch_value in sorted(unique_pitches)
}
duration_stream = (event["duration_beat"] for event in NOTE_EVENTS)

assert long_pitches_loop == long_pitches
print(f"达到阈值的音高：{long_pitches}")
print(f"各音高出现次数：{pitch_counts}")
print(f"生成器产生的时值总和：{sum(duration_stream):.2f} 拍")

## 5. 函数、参数与返回值

函数把一组操作封装起来，调用时可以根据输入计算结果。定义函数时列出形参，调用函数时传入实参；`return` 结束当前函数并返回结果。在函数体内赋值的名称默认属于局部作用域。函数体开头单独写的一段字符串，会被 Python 记录为函数的文档字符串（docstring），可通过 `help()` 查看。

下面的 `pitch: int` 和 `-> int` 是类型注解，可帮助编辑器和静态检查工具分析代码，也可明确接口预期，但不会自行在运行时拒绝错误类型。默认值在函数定义时求值；列表、字典等可变对象若直接作为默认值，会被后续调用共享。

In [ ]:
def transpose_pitch(pitch: int, semitones: int = 0) -> int:
    """返回按指定半音数移调后的 MIDI 音高。"""
    transposed = pitch + semitones
    if not 0 <= transposed <= 127:
        raise ValueError("移调结果必须在 MIDI 音高 0 至 127 之间")
    return transposed


def pitch_span(events):
    """返回非空事件序列中的最低音与最高音。"""
    pitches = [event["pitch_midi"] for event in events]
    return min(pitches), max(pitches)


shifted_pitch = transpose_pitch(60, semitones=TRANSPOSE_SEMITONES)
lowest_pitch, highest_pitch = pitch_span(NOTE_EVENTS)
print(f"移调结果：{shifted_pitch}")
print(f"音域：{lowest_pitch}—{highest_pitch}，跨度 {highest_pitch - lowest_pitch} 个半音")

调用表达式中的 `**mapping` 把映射的字符串键作为关键字名、对应值作为实参传入；函数定义中的 `**kwargs` 则把未被其他形参接收的关键字实参汇集为字典。

In [ ]:
def format_event(pitch, duration, unit="拍"):
    """把一组音高与时值格式化为显示文本。"""
    return f"MIDI {pitch}，{duration:.2f} {unit}"


display_options = {"unit": "拍"}
print(format_event(60, 1.0, **display_options))


def collect_named_options(**kwargs):
    """把额外的命名参数作为字典返回，供认读结构。"""
    return kwargs


print(collect_named_options(color="black", marker="o"))

## 6. 模块、对象与警告

`import os` 导入模块并绑定名称 `os`；`import numpy as np` 导入模块并绑定别名 `np`；`from pathlib import Path` 则从模块中导入名称 `Path`。`os`、`sys`、`pathlib` 属于标准库；NumPy 等第三方库通常需要另行安装；以 `_common` 为例，项目模块来自配套代码目录。包管理工具使用的安装名与代码中的导入名可能不同，例如安装 `scikit-learn` 后写 `import sklearn`。

类定义对象的结构和行为，实例是按某个类创建的具体对象。`Path` 是类，`REQUIREMENTS_PATH = Path(...)` 创建路径对象；`.name`、`.parent` 读取属性，`.exists()`、`.resolve()` 则调用方法。形如 `note.pitch` 的表达式读取属性，`pm.get_end_time()` 则调用方法。

In [ ]:
REQUIREMENTS_PATH = CHAPTER_DIR / "requirements.txt"
print(f"对象类型：{type(REQUIREMENTS_PATH).__name__}")
print(f"文件名属性：{REQUIREMENTS_PATH.name}")
print(f"父目录属性：{REQUIREMENTS_PATH.parent}")
print(f"文件存在：{REQUIREMENTS_PATH.exists()}")
print(f"解析后的路径：{REQUIREMENTS_PATH.resolve()}")

警告通常不会立即终止程序，但会提示值得检查的数据条件、接口变化或运行环境。过滤时应先理解原因，再限定消息模式、警告类别和影响范围。下面只过滤一条由本单元主动产生、消息模式和类别均匹配的示例警告。

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Chapter 1 targeted warning",
        category=UserWarning,
    )
    warnings.warn("Chapter 1 targeted warning", UserWarning)

print("仅匹配指定消息和类别的示例警告被过滤。")

## 7. 路径与文本文件

`pathlib.Path` 以对象表示文件系统路径。`/` 在这里用于构造子路径，不是除法；`.exists()` 检查文件或目录是否存在；`.resolve()` 返回绝对路径，并解析符号链接和 `..` 等路径成分。读取文本时显式给出编码，并用 `with` 保证退出代码块时关闭文件。

In [ ]:
if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(f"缺少文件：{REQUIREMENTS_PATH}")

with REQUIREMENTS_PATH.open("r", encoding="utf-8") as file:
    first_line = file.readline().strip()

print(f"依赖文件第一行：{first_line}")

同一组路径操作也可以使用标准库中的 `os.path` 表达，几项常用写法对应如下：

| `os` / `os.path` 写法 | `Path` 写法 | 含义 |
|---|---|---|
| `os.getcwd()` | `Path.cwd()` | 当前工作目录 |
| `os.path.join(a, b)` | `Path(a) / b` | 拼接路径 |
| `os.path.exists(p)` | `Path(p).exists()` | 检查是否存在 |
| `os.path.dirname(p)` | `Path(p).parent` | 取得父目录 |

In [ ]:
old_style_path = os.path.join(str(PROJECT_ROOT), "CODE", "chapter01", "requirements.txt")
new_style_path = PROJECT_ROOT / "CODE" / "chapter01" / "requirements.txt"

print(f"os.getcwd()：{os.getcwd()}")
print(f"os.path.exists(...)：{os.path.exists(old_style_path)}")
print(f"os.path.dirname(...)：{os.path.dirname(old_style_path)}")
print(f"两种写法指向同一位置：{Path(old_style_path) == new_style_path}")

部分后续 Notebook 会导入项目内的 `_common` 包。代码先定位项目根目录，再把包含该包的目录加入 `sys.path`，使导入系统能够搜索到项目模块。这项操作改变的是模块搜索路径，不能代替在当前环境中安装第三方包。

## 8. 异常与回溯信息

异常传播到解释器且未被处理时，Python 通常会打印异常回溯（traceback）。先读最后一行，判断异常类型和具体消息；再从末尾向上寻找最接近出错位置、且属于自己代码的调用位置。常见类型包括：

| 异常 | 常见含义 | 初步核对点 |
|---|---|---|
| `NameError` | 名称尚未定义 | 拼写与单元格执行顺序 |
| `TypeError` | 操作或参数类型不合适 | `type(...)` 与函数签名 |
| `ValueError` | 类型可以接受，但值或格式不合要求 | 取值范围、字符串格式与异常消息 |
| `IndexError` | 序列索引越界 | `len(...)` 与索引范围 |
| `KeyError` | 映射中没有该键 | 实际键名、拼写与 `.keys()` |
| `FileNotFoundError` | 请求的文件或目录不存在 | cwd、解析后的路径与 `.exists()` |
| `ModuleNotFoundError` | 导入系统未找到所请求的模块 | 缺少的模块名、`sys.executable` 与安装环境 |

示例函数只捕获事先声明的预期异常；其他错误仍会向外传播，不会被隐藏。

In [ ]:
def show_expected_error(label, expected_error, action):
    """运行一个操作并打印预期异常，不隐藏其他错误。"""
    try:
        action()
    except expected_error as error:
        print(f"{label} -> {type(error).__name__}: {error}")


def use_undefined_name():
    return pitch_not_defined


missing_file = CHAPTER_DIR / "file_that_does_not_exist.txt"
show_expected_error("未定义名称", NameError, use_undefined_name)
show_expected_error("数值与字符串相加", TypeError, lambda: 60 + "1")
show_expected_error("列表索引越界", IndexError, lambda: [60, 62][3])
show_expected_error("字典缺少键", KeyError, lambda: {"pitch": 60}["duration"])
show_expected_error("文件不存在", FileNotFoundError, lambda: missing_file.read_text(encoding="utf-8"))
show_expected_error(
    "模块不存在",
    ModuleNotFoundError,
    lambda: importlib.import_module("chapter01_package_that_does_not_exist"),
)
show_expected_error("字符串不能转为整数", ValueError, lambda: int("C4"))

一段典型错误的末尾可能是：

```text
Traceback (most recent call last):
  File "...", line 1, in <module>
    Path("data/melody.csv").read_text(encoding="utf-8")
FileNotFoundError: [Errno 2] No such file or directory: 'data/melody.csv'
```

最后一行指向文件或路径问题，应先打印 `Path.cwd()`、解析后的完整路径和 `.exists()`，文件路径问题与是否安装某个包无关，无需据此安装 Pandas。若是 `ModuleNotFoundError`，则先读取缺少的模块名，核对 `sys.executable`，再检查该解释器中的安装状态。数据运算报错时，还应查看输入的 `type`、取值和 `shape`。

`try/except` 只应捕获程序知道如何处理、补充说明或恢复的异常，未预料的异常应继续向外传播。`raise` 用于主动抛出异常，例如报告输入不满足约束。`assert` 用于检查程序内部应当成立的不变量；以 `python -O`（大写字母 O）启动解释器时，`assert` 不会生成可执行代码，因此外部输入和文件存在性应通过显式检查处理。避免使用 `except Exception: pass`，它会丢弃大多数常规运行时异常，使程序可能带着不完整或错误的结果继续执行。

In [ ]:
validated_pitch = transpose_pitch(60, semitones=12)
assert 0 <= validated_pitch <= 127
print(f"内部条件成立：移调结果为 {validated_pitch}")

try:
    transpose_pitch(127, semitones=1)
except ValueError as error:
    print(f"输入验证报告：{error}")

## 小结

以上示例以同一组音符事件串联了基础类型、容器、流程控制、函数、模块和路径操作。排查错误时，先根据异常回溯最后一行判断问题属于名称、类型、取值、索引、路径还是导入环境，再核对与该类问题直接相关的信息。下一份 Notebook 将把同一组事件转换为数组和表格，并绘制三种图形。